# 运行时与方法机制预检入口

该 Notebook 用于 Colab 诊断和机制预检: 挂载 Google Drive, 拉取仓库代码, 安装 Colab 默认可升级依赖, 登录 Hugging Face, 加载真实 SD3.5 Medium 主模型, 捕获真实 latent trajectory, 再执行最小 latent injection 机制闭环。

运行依赖: 不依赖任何正式结果包, 需要 GPU, 可在正式运行前独立执行。该入口合并原运行时诊断与最小机制预检, 不参与 `pilot_paper` 或 `full_paper` 的正式统计产出。

正式逻辑分别位于 `paper_workflow/colab_utils/sd_runtime_cold_start.py` 与 `paper_workflow/colab_utils/minimal_latent_injection.py`, Notebook 只作为远程执行入口。

## 运行前准备

1. 在 Colab 中选择 GPU runtime。
2. 准备 Hugging Face token, 并确认账号已获得目标模型访问权限。
3. 确认 Google Drive 可挂载。默认保存目录为 `/content/drive/MyDrive/SLM/runtime_method_precheck/`。
4. 默认 `SLM_WM_RUNTIME_MODEL_SELECTION=auto` 且 `SLM_WM_INJECTION_MODEL_SELECTION=auto`, 即只运行 SD3.5 Medium 主模型。若需要 SD3 Medium 兼容对照, 可在运行前显式设置 runtime 选择为 `both`。

In [ ]:
SLM_WM_PAPER_RUN_NAME = "pilot_paper"

import os
os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME

from google.colab import drive

drive.mount('/content/drive')
os.environ.setdefault('SLM_WM_RUNTIME_MODEL_SELECTION', 'auto')
os.environ.setdefault('SLM_WM_INJECTION_MODEL_SELECTION', 'auto')
os.environ.setdefault('SLM_WM_PRECHECK_DRIVE_ROOT', '/content/drive/MyDrive/SLM/runtime_method_precheck')
os.environ.setdefault(
    'SLM_WM_RUNTIME_DRIVE_OUTPUT_DIR',
    os.path.join(os.environ['SLM_WM_PRECHECK_DRIVE_ROOT'], 'real_sd_runtime_probe'),
)
os.environ.setdefault(
    'SLM_WM_INJECTION_DRIVE_OUTPUT_DIR',
    os.path.join(os.environ['SLM_WM_PRECHECK_DRIVE_ROOT'], 'minimal_diffusion_latent_injection'),
)
print({
    'runtime_model_selection': os.environ['SLM_WM_RUNTIME_MODEL_SELECTION'],
    'injection_model_selection': os.environ['SLM_WM_INJECTION_MODEL_SELECTION'],
    'runtime_drive_output_dir': os.environ['SLM_WM_RUNTIME_DRIVE_OUTPUT_DIR'],
    'injection_drive_output_dir': os.environ['SLM_WM_INJECTION_DRIVE_OUTPUT_DIR'],
})


In [ ]:
%pip install -q --upgrade diffusers transformers accelerate safetensors sentencepiece protobuf huggingface_hub


In [ ]:
# 依赖诊断已收敛到 paper_workflow.colab_utils.dependency_check, 在仓库 checkout 后统一执行。


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})

from paper_workflow.colab_utils.dependency_check import build_notebook_dependency_report
dependency_report = build_notebook_dependency_report("sd35_runtime")
print(dependency_report)


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get('HF_TOKEN')
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = token_from_secret

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('huggingface_login_ready')
else:
    print('HF_TOKEN_not_found')


In [ ]:
import torch

device_report = {
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
print(device_report)
assert device_report['cuda_available'], '需要 Colab GPU runtime'


In [ ]:
import os

from paper_workflow.colab_utils.sd_runtime_cold_start import run_default_model_plan

runtime_model_selection = os.environ.get('SLM_WM_RUNTIME_MODEL_SELECTION', 'auto')
runtime_summaries = run_default_model_plan(root='.', model_selection=runtime_model_selection)
runtime_summaries


In [ ]:
import os

from paper_workflow.colab_utils.minimal_latent_injection import run_default_injection_plan

injection_model_selection = os.environ.get('SLM_WM_INJECTION_MODEL_SELECTION', 'auto')
injection_summaries = run_default_injection_plan(root='.', model_selection=injection_model_selection)
injection_summaries


In [ ]:
from paper_workflow.colab_utils.notebook_entrypoint import package_runtime_method_precheck_outputs

archive_records = package_runtime_method_precheck_outputs(root='.')
archive_records


In [ ]:
from pathlib import Path

for summary_path in sorted(Path('outputs/real_sd_runtime_probe').glob('*_summary.json')):
    print(summary_path)
    print(summary_path.read_text(encoding='utf-8'))

for result_path in sorted(Path('outputs/minimal_diffusion_latent_injection').glob('*_injection_result.json')):
    print(result_path)
    print(result_path.read_text(encoding='utf-8'))
